# Building a RAG system with LangChain and ChromaDB

### Introduction

##### Retrieval Augmented Generation (RAG) is a powerful technique that combines the capabilities of large language models with external knowledge retrieval. This notebook will walk you through building a complete RAG system using 

- LangChain : A framework for developing applications powered by language models.

- ChromaDB : An open-source vector database for storing and retrieving embeddings.

- OpenAI : For embeddings and language model(you can substitute with other providers).

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
# langchain imports
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document

# vectorstores imports
from langchain_community.vectorstores import Chroma

# utility imports
import numpy as np
from typing import List


d:\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\vaibh\AppData\Local\Temp\ipykernel_15688\3894926957.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


### RAG (Retrieval Augmented Generation) Architechture

1. Document Loading : Load documents from various sources
2. Document Splitting : Break documents into smaller chunks
3. Embedding Generation : Convert chunks into vector representation
4. Vector Storage : Store embeddings into ChromaDB
5. Query Processing : Convert user query to embedding
6. Similarity Search : Find relevant chunks from the vector store
7. Context Augmentation : Combine retrieved chunks with query
8. Response Generation : LLM generates answer using context

###### Benefits of RAG

- reduce hallucations
- provides up to date information
- allows citing sources
- work with domain specific knowledge

#### 1. Sample Data

In [3]:
# create sample documents

sample_docs = [

    """Machine Learning (ML)

Machine Learning (ML) is a branch of Artificial Intelligence (AI) that enables computers to learn patterns from data and make decisions or predictions without being explicitly programmed for every task. Instead of following fixed rules, a machine learning model improves its performance by analyzing historical data and identifying relationships within it. ML is widely used in applications such as spam email detection, fraud detection, recommendation systems, customer segmentation, predictive analytics, and demand forecasting.""",
"""Deep Learning (DL)

Deep Learning is a specialized subset of Machine Learning that uses artificial neural networks with multiple hidden layers to learn complex patterns from large volumes of data. These neural networks are inspired by the structure and functioning of the human brain, where interconnected neurons process information and learn from experience. Unlike traditional machine learning models, deep learning automatically learns important features from raw data, eliminating much of the need for manual feature engineering.""",
"""Natural Language Processing (NLP)

Natural Language Processing (NLP) is a field of Artificial Intelligence that focuses on enabling computers to understand, interpret, process, and generate human language. The primary objective of NLP is to bridge the communication gap between humans and machines by allowing computers to work with text and speech in a meaningful way. NLP combines concepts from computer science, machine learning, deep learning, and linguistics to analyze natural language."""
]

sample_docs

['Machine Learning (ML)\n\nMachine Learning (ML) is a branch of Artificial Intelligence (AI) that enables computers to learn patterns from data and make decisions or predictions without being explicitly programmed for every task. Instead of following fixed rules, a machine learning model improves its performance by analyzing historical data and identifying relationships within it. ML is widely used in applications such as spam email detection, fraud detection, recommendation systems, customer segmentation, predictive analytics, and demand forecasting.',
 'Deep Learning (DL)\n\nDeep Learning is a specialized subset of Machine Learning that uses artificial neural networks with multiple hidden layers to learn complex patterns from large volumes of data. These neural networks are inspired by the structure and functioning of the human brain, where interconnected neurons process information and learn from experience. Unlike traditional machine learning models, deep learning automatically lea

In [4]:
# save the documnets to a file

import tempfile
temp_dir = tempfile.mkdtemp()

for i,doc in enumerate(sample_docs):
    with open(f"{temp_dir}/doc_{i}.txt","w") as f:
        f.write(doc)

print(f"sample document created in {temp_dir}")

sample document created in C:\Users\vaibh\AppData\Local\Temp\tmpbf3opwa9


In [5]:
# save the documnets to a file

import tempfile
temp_dir = tempfile.mkdtemp()

for i,doc in enumerate(sample_docs):
    with open(f"doc_{i}.txt","w") as f:
        f.write(doc)


#### 2. Documnet Loading

In [6]:
temp_dir

'C:\\Users\\vaibh\\AppData\\Local\\Temp\\tmpu8gxnbqe'

In [7]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader

# load directory from directory
loader = DirectoryLoader(
    "data",
    glob="*.txt",
    loader_cls=TextLoader,
    loader_kwargs={'encoding':'utf-8'}
)

documents = loader.load()

print("loaded - ",len(documents),"documents")
print("first document preview:")
print(documents[0].page_content[0:200],"...")

loaded -  3 documents
first document preview:
Machine Learning (ML)

Machine Learning (ML) is a branch of Artificial Intelligence (AI) that enables computers to learn patterns from data and make decisions or predictions without being explicitly p ...


#### 3. Document Splitting

In [8]:
documents

[Document(metadata={'source': 'data\\doc_0.txt'}, page_content='Machine Learning (ML)\n\nMachine Learning (ML) is a branch of Artificial Intelligence (AI) that enables computers to learn patterns from data and make decisions or predictions without being explicitly programmed for every task. Instead of following fixed rules, a machine learning model improves its performance by analyzing historical data and identifying relationships within it. ML is widely used in applications such as spam email detection, fraud detection, recommendation systems, customer segmentation, predictive analytics, and demand forecasting.'),
 Document(metadata={'source': 'data\\doc_1.txt'}, page_content='Deep Learning (DL)\n\nDeep Learning is a specialized subset of Machine Learning that uses artificial neural networks with multiple hidden layers to learn complex patterns from large volumes of data. These neural networks are inspired by the structure and functioning of the human brain, where interconnected neuro

In [9]:
recursive_splitter = RecursiveCharacterTextSplitter(
    # separators=["\n\n", "\n", " ", ""], # try this seperators in order
    separators=[" "], # try this seperators in order
    chunk_size = 200,
    chunk_overlap = 20,
    length_function = len
)

recursive_chunks = recursive_splitter.split_documents(documents)

print("created",len(recursive_chunks)," chunks from a",len(documents),' documents')
print("chunk example :")
print("content = ",recursive_chunks[0].page_content[0:200],"...")
print("metadata = ",recursive_chunks[0].metadata)
print()
print('recursive splitter implemented successfully')

created 9  chunks from a 3  documents
chunk example :
content =  Machine Learning (ML)

Machine Learning (ML) is a branch of Artificial Intelligence (AI) that enables computers to learn patterns from data and make decisions or predictions without being explicitly ...
metadata =  {'source': 'data\\doc_0.txt'}

recursive splitter implemented successfully


#### 4. Data Embedding

In [10]:
sample_text = "machine learning is Amazing!"
embeddings = OpenAIEmbeddings()
embeddings


OpenAIEmbeddings(client=<openai.resources.embeddings.Embeddings object at 0x000002166B04A660>, async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x000002166B049EE0>, model='text-embedding-ada-002', dimensions=None, deployment='text-embedding-ada-002', openai_api_version=None, openai_api_base=None, openai_api_type=None, openai_proxy=None, embedding_ctx_length=8191, openai_api_key=SecretStr('**********'), openai_organization=None, allowed_special=None, disallowed_special=None, chunk_size=1000, max_retries=2, request_timeout=None, headers=None, tiktoken_enabled=True, tiktoken_model_name=None, show_progress_bar=False, model_kwargs={}, skip_empty=False, default_headers=None, default_query=None, retry_min_seconds=4, retry_max_seconds=20, http_client=None, http_async_client=None, check_embedding_ctx_length=True)

In [11]:
vector = embeddings.embed_query(sample_text)
vector

[-0.030300041660666466,
 0.00174097646959126,
 0.013390576466917992,
 -0.0252634659409523,
 -0.008589041419327259,
 0.01216165255755186,
 0.012181798927485943,
 0.000823060458060354,
 -0.01786305569112301,
 -0.03800936043262482,
 0.006937044207006693,
 0.02968222089111805,
 -0.012349684722721577,
 -0.014317306689918041,
 0.008837511762976646,
 0.025894716382026672,
 0.024565059691667557,
 0.006631491705775261,
 0.0029682221356779337,
 -0.006772515829652548,
 -0.03094472363591194,
 0.0365319661796093,
 0.010711118578910828,
 -0.03473223000764847,
 -0.022308673709630966,
 0.004821682348847389,
 0.00577191635966301,
 -0.038036223500967026,
 -0.011402808129787445,
 -0.017889918759465218,
 0.027385542169213295,
 -0.005422713700681925,
 -0.014867972582578659,
 -0.026257349178195,
 -0.020938726142048836,
 -0.0017359398771077394,
 0.003881521290168166,
 0.005788704846054316,
 0.00015581907064188272,
 -0.0040829842910170555,
 0.024068117141723633,
 0.018655477091670036,
 0.0018584965728223324,


#### Initialize the ChromaDB Vector Store & stores the chunks Vector Representation

In [12]:
# create a ChromaDB vector store
persist_directory = "./chroma_db"

# initialize chromadb with OpenAI embeddings
vectorStore = Chroma.from_documents(
    documents=recursive_chunks,
    embedding=OpenAIEmbeddings(),
    persist_directory=persist_directory,
    collection_name="rag_collection"
)

print("vector store created with ",vectorStore._collection.count(),"vectors")
print("persistent_directory = ",persist_directory)

vector store created with  36 vectors
persistent_directory =  ./chroma_db


#### Test Similarity Search

In [13]:
query = "what are the types of machine learning?"

similar_docs = vectorStore.similarity_search(query,k=3)
similar_docs

[Document(metadata={'source': 'data\\doc_0.txt'}, page_content='Machine Learning (ML)\n\nMachine Learning (ML) is a branch of Artificial Intelligence (AI) that enables computers to learn patterns from data and make decisions or predictions without being explicitly'),
 Document(metadata={'source': 'data\\doc_0.txt'}, page_content='Machine Learning (ML)\n\nMachine Learning (ML) is a branch of Artificial Intelligence (AI) that enables computers to learn patterns from data and make decisions or predictions without being explicitly'),
 Document(metadata={'source': 'data\\doc_0.txt'}, page_content='Machine Learning (ML)\n\nMachine Learning (ML) is a branch of Artificial Intelligence (AI) that enables computers to learn patterns from data and make decisions or predictions without being explicitly')]

In [14]:
print(f"query = {query}")
print(f"top {len(similar_docs)} similar chunks : ")

for i,doc in enumerate(similar_docs):
    print(f"chunk {i+1}")
    print(doc.page_content)
    print(f"source = {doc.metadata.get('source','unknown')}")

query = what are the types of machine learning?
top 3 similar chunks : 
chunk 1
Machine Learning (ML)

Machine Learning (ML) is a branch of Artificial Intelligence (AI) that enables computers to learn patterns from data and make decisions or predictions without being explicitly
source = data\doc_0.txt
chunk 2
Machine Learning (ML)

Machine Learning (ML) is a branch of Artificial Intelligence (AI) that enables computers to learn patterns from data and make decisions or predictions without being explicitly
source = data\doc_0.txt
chunk 3
Machine Learning (ML)

Machine Learning (ML) is a branch of Artificial Intelligence (AI) that enables computers to learn patterns from data and make decisions or predictions without being explicitly
source = data\doc_0.txt


In [15]:
query2 = "what is NLP?"

similar_docs2 = vectorStore.similarity_search(query2,k=3)
similar_docs2

[Document(metadata={'source': 'data\\doc_2.txt'}, page_content='meaningful way. NLP combines concepts from computer science, machine learning, deep learning, and linguistics to analyze natural language.'),
 Document(metadata={'source': 'data\\doc_2.txt'}, page_content='meaningful way. NLP combines concepts from computer science, machine learning, deep learning, and linguistics to analyze natural language.'),
 Document(metadata={'source': 'data\\doc_2.txt'}, page_content='meaningful way. NLP combines concepts from computer science, machine learning, deep learning, and linguistics to analyze natural language.')]

In [16]:
print(f"query = {query2}")
print(f"top {len(similar_docs2)} similar chunks : ")

for i,doc in enumerate(similar_docs2):
    print(f"chunk {i+1}")
    print(doc.page_content)
    print(f"source = {doc.metadata.get('source','unknown')}")

query = what is NLP?
top 3 similar chunks : 
chunk 1
meaningful way. NLP combines concepts from computer science, machine learning, deep learning, and linguistics to analyze natural language.
source = data\doc_2.txt
chunk 2
meaningful way. NLP combines concepts from computer science, machine learning, deep learning, and linguistics to analyze natural language.
source = data\doc_2.txt
chunk 3
meaningful way. NLP combines concepts from computer science, machine learning, deep learning, and linguistics to analyze natural language.
source = data\doc_2.txt


In [17]:
query3 = "what is deep learning?"

similar_docs3 = vectorStore.similarity_search(query3,k=3)
similar_docs3

[Document(metadata={'source': 'data\\doc_1.txt'}, page_content='Deep Learning (DL)\n\nDeep Learning is a specialized subset of Machine Learning that uses artificial neural networks with multiple hidden layers to learn complex patterns from large volumes of data.'),
 Document(metadata={'source': 'data\\doc_1.txt'}, page_content='Deep Learning (DL)\n\nDeep Learning is a specialized subset of Machine Learning that uses artificial neural networks with multiple hidden layers to learn complex patterns from large volumes of data.'),
 Document(metadata={'source': 'data\\doc_1.txt'}, page_content='Deep Learning (DL)\n\nDeep Learning is a specialized subset of Machine Learning that uses artificial neural networks with multiple hidden layers to learn complex patterns from large volumes of data.')]

In [18]:
print(f"query = {query3}")
print(f"top {len(similar_docs3)} similar chunks : ")

for i,doc in enumerate(similar_docs3):
    print(f"chunk {i+1}")
    print(doc.page_content)
    print(f"source = {doc.metadata.get('source','unknown')}")

query = what is deep learning?
top 3 similar chunks : 
chunk 1
Deep Learning (DL)

Deep Learning is a specialized subset of Machine Learning that uses artificial neural networks with multiple hidden layers to learn complex patterns from large volumes of data.
source = data\doc_1.txt
chunk 2
Deep Learning (DL)

Deep Learning is a specialized subset of Machine Learning that uses artificial neural networks with multiple hidden layers to learn complex patterns from large volumes of data.
source = data\doc_1.txt
chunk 3
Deep Learning (DL)

Deep Learning is a specialized subset of Machine Learning that uses artificial neural networks with multiple hidden layers to learn complex patterns from large volumes of data.
source = data\doc_1.txt


##### Advanced Similarity Search with scores

In [19]:
result_score = vectorStore.similarity_search_with_score(query,k=3)
result_score

[(Document(metadata={'source': 'data\\doc_0.txt'}, page_content='Machine Learning (ML)\n\nMachine Learning (ML) is a branch of Artificial Intelligence (AI) that enables computers to learn patterns from data and make decisions or predictions without being explicitly'),
  0.3134913444519043),
 (Document(metadata={'source': 'data\\doc_0.txt'}, page_content='Machine Learning (ML)\n\nMachine Learning (ML) is a branch of Artificial Intelligence (AI) that enables computers to learn patterns from data and make decisions or predictions without being explicitly'),
  0.3134913444519043),
 (Document(metadata={'source': 'data\\doc_0.txt'}, page_content='Machine Learning (ML)\n\nMachine Learning (ML) is a branch of Artificial Intelligence (AI) that enables computers to learn patterns from data and make decisions or predictions without being explicitly'),
  0.3134913444519043)]

In [20]:
result_score2 = vectorStore.similarity_search_with_score(query2,k=3)
result_score2

[(Document(metadata={'source': 'data\\doc_2.txt'}, page_content='meaningful way. NLP combines concepts from computer science, machine learning, deep learning, and linguistics to analyze natural language.'),
  0.2654258906841278),
 (Document(metadata={'source': 'data\\doc_2.txt'}, page_content='meaningful way. NLP combines concepts from computer science, machine learning, deep learning, and linguistics to analyze natural language.'),
  0.2654258906841278),
 (Document(metadata={'source': 'data\\doc_2.txt'}, page_content='meaningful way. NLP combines concepts from computer science, machine learning, deep learning, and linguistics to analyze natural language.'),
  0.2654258906841278)]

In [21]:
result_score3 = vectorStore.similarity_search_with_score(query3,k=3)
result_score3

[(Document(metadata={'source': 'data\\doc_1.txt'}, page_content='Deep Learning (DL)\n\nDeep Learning is a specialized subset of Machine Learning that uses artificial neural networks with multiple hidden layers to learn complex patterns from large volumes of data.'),
  0.2118757665157318),
 (Document(metadata={'source': 'data\\doc_1.txt'}, page_content='Deep Learning (DL)\n\nDeep Learning is a specialized subset of Machine Learning that uses artificial neural networks with multiple hidden layers to learn complex patterns from large volumes of data.'),
  0.2118757665157318),
 (Document(metadata={'source': 'data\\doc_1.txt'}, page_content='Deep Learning (DL)\n\nDeep Learning is a specialized subset of Machine Learning that uses artificial neural networks with multiple hidden layers to learn complex patterns from large volumes of data.'),
  0.2118757665157318)]

##### Initialize LLM, RAG chain, Prompt Template, Query the RAG system

In [22]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-3.5-turbo"
    # temperature=0.2,
    # max_completion_tokens=500
)

In [23]:
test_response = llm.invoke("what is LLM?")
test_response

AIMessage(content='LLM stands for Master of Laws, which is an advanced law degree typically pursued by individuals who already have a law degree and want to specialize in a particular area of law or gain expertise in a specific legal topic.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 43, 'prompt_tokens': 12, 'total_tokens': 55, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-E3xOJzMzWZl4GOykalV3BFWLnCAr5', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f8329-ec8c-7f43-ac4e-e64daee2cd87-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 12, 'output_tokens': 43, 'total_tokens': 55, 'input_token_details': {'audio': 0

In [24]:
from langchain.chat_models.base import init_chat_model

llm2 = init_chat_model("openai:gpt-3.5-turbo")
#llm3 = init_chat_model("groq:")

llm2

ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11', 'langchain-openai': '1.3.3'}}, output_version=None, profile={'name': 'GPT-3.5-turbo', 'release_date': '2023-03-01', 'last_updated': '2023-11-06', 'open_weights': False, 'max_input_tokens': 16385, 'max_output_tokens': 4096, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': False, 'structured_output': False, 'attachment': False, 'temperature': True, 'image_url_inputs': False, 'pdf_inputs': False, 'pdf_tool_message': False, 'image_tool_message': False, 'tool_choice': True, 'tool_call_streaming': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x000002166EA25FD0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000002166EA26BA0>, root_client=<openai.OpenAI

In [25]:
test_response2 = llm2.invoke("what is AI?")
test_response2

AIMessage(content='AI, or artificial intelligence, refers to the technology that enables machines to think, learn, and perform tasks that normally require human intelligence. AI algorithms allow computers to analyze data, make decisions, and solve problems in ways that mimic human cognitive functions. Some common applications of AI include speech and image recognition, natural language processing, autonomous vehicles, and predictive analytics.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 71, 'prompt_tokens': 11, 'total_tokens': 82, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-E3xOLDSD0oqknQIiK1A4uersTjWPZ', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': No

#### Modern RAG chain

In [27]:
from langchain_classic.chains import create_retrieval_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

In [28]:
##### convert vector store into retriever

retriever = vectorStore.as_retriever(
    search_kwarg = {"k":3} # retrive top 3 relevant chunk
)

retriever

VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x000002166B9AEA50>, search_kwargs={})

In [29]:
# create a prompt template

from langchain_core.prompts import ChatPromptTemplate

system_prompt = """you are an assistant for question answering tasks.
Used a following pieces of retrieved context to answer the question.
If you don't know the answer, just say you don't know.
Use three sentences maximum and keep the answer concise.

context : {context}"""

prompt = ChatPromptTemplate.from_messages([
    ("system",system_prompt),
    ("human","{input}")
])

In [30]:
prompt

ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="you are an assistant for question answering tasks.\nUsed a following pieces of retrieved context to answer the question.\nIf you don't know the answer, just say you don't know.\nUse three sentences maximum and keep the answer concise.\n\ncontext : {context}"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])

In [31]:
import langchain
print(langchain.__version__)

1.3.11


In [32]:
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain

In [33]:
# create a document chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

document_chain = create_stuff_documents_chain(llm,prompt)
document_chain


RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="you are an assistant for question answering tasks.\nUsed a following pieces of retrieved context to answer the question.\nIf you don't know the answer, just say you don't know.\nUse three sentences maximum and keep the answer concise.\n\ncontext : {context}"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])
| ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11', 'langchain-openai': '1.3.3'}}, output_version=None, p

#### create stuff documets chain

- takes retrieved documents
- stuffs them into the prompt's {context} placeholder.
- sends the complete prompt to the LLM.
- returns the LLM's response.


ChatPromptTemplate - prompt
ChatOpenAI - LLM
StrOutputParser() - how your output is going to get
RunnableBinding - run entire chain one by one

In [34]:
# create rag chain

rag_chain = create_retrieval_chain(retriever,document_chain)
rag_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x000002166B9AEA50>, search_kwargs={}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="you are an assistant for question answering tasks.\nUsed a following pieces of retrieved context to answer the question.\nIf you don't k

RunnnableBinding

retriever - will fetch the relevant information
document chain - will process documents with the LLM.


In [36]:
response = rag_chain.invoke({"input":"what is deep learning"})
response

{'input': 'what is deep learning',
 'context': [Document(metadata={'source': 'data\\doc_1.txt'}, page_content='Deep Learning (DL)\n\nDeep Learning is a specialized subset of Machine Learning that uses artificial neural networks with multiple hidden layers to learn complex patterns from large volumes of data.'),
  Document(metadata={'source': 'data\\doc_1.txt'}, page_content='Deep Learning (DL)\n\nDeep Learning is a specialized subset of Machine Learning that uses artificial neural networks with multiple hidden layers to learn complex patterns from large volumes of data.'),
  Document(metadata={'source': 'data\\doc_1.txt'}, page_content='Deep Learning (DL)\n\nDeep Learning is a specialized subset of Machine Learning that uses artificial neural networks with multiple hidden layers to learn complex patterns from large volumes of data.'),
  Document(metadata={'source': 'data\\doc_1.txt'}, page_content='Deep Learning (DL)\n\nDeep Learning is a specialized subset of Machine Learning that use

In [37]:
response['answer']

'Deep Learning is a subset of Machine Learning that uses artificial neural networks with multiple hidden layers to learn complex patterns from data.'

In [38]:
#

def query_rag_fun(question):
    print(f"question : {question}")
    print("-" * 50)

    # using create_retrieval_chain approach
    result = rag_chain.invoke({"input":question})

    print(f"answer:{result['answer']}")
    print("\nretrieved content : ")
    for i,doc in enumerate(result['context']):
        print(f"\n -- source {i+1} --")
        print(doc.page_content[:200] + "...")
    
    return result

test_questions = [
    "what are the three types of machine learning?",
    "what is deep learning and how it relate te neural networka?",
    "what are CNN best used for?"
]

for question in test_questions:
    result = query_rag_fun(question)
    print("\n" + "="*80 + "\n")

question : what are the three types of machine learning?
--------------------------------------------------
answer:The three types of machine learning are supervised learning, unsupervised learning, and reinforcement learning.

retrieved content : 

 -- source 1 --
Machine Learning (ML)

Machine Learning (ML) is a branch of Artificial Intelligence (AI) that enables computers to learn patterns from data and make decisions or predictions without being explicitly...

 -- source 2 --
Machine Learning (ML)

Machine Learning (ML) is a branch of Artificial Intelligence (AI) that enables computers to learn patterns from data and make decisions or predictions without being explicitly...

 -- source 3 --
Machine Learning (ML)

Machine Learning (ML) is a branch of Artificial Intelligence (AI) that enables computers to learn patterns from data and make decisions or predictions without being explicitly...

 -- source 4 --
Machine Learning (ML)

Machine Learning (ML) is a branch of Artificial Intell